# NeuroMera - Emotion Classification (BERT)

**How to use this notebook:**
1. Make sure GPU is enabled: `Runtime > Change runtime type > T4 GPU`
2. Upload your CSV to Google Drive (must have two columns: `text` and `label`)
3. Update the `CSV_PATH` in Step 1 below
4. Run all cells top to bottom: `Runtime > Run all`
5. Your trained model will be saved to Google Drive

**Labels your CSV should have (exactly these 6):**
`sadness`, `love`, `anger`, `joy`, `fear`, `surprise`

---
## Step 1 - Setup & Load Data
Update `CSV_PATH` to point to your dataset.

In [ ]:
# Install dependencies
!pip install -q transformers accelerate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import torch
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
#  CHANGE THIS to your CSV path on Google Drive
# ============================================================
CSV_PATH = "/content/drive/MyDrive/FYPTextData/long_clean_balanced_dataset.csv"

# Where to save the trained model on Drive (created automatically)
MODEL_DIR = "/content/drive/MyDrive/NeuroMera_Model"
# ============================================================

df = pd.read_csv(CSV_PATH)
df.columns = ["text", "label"]
df = df.dropna().drop_duplicates(subset=["text"]).reset_index(drop=True)

print(f"Loaded {len(df)} rows")
print(f"Labels: {sorted(df['label'].unique())}")
print(f"\nDistribution:\n{df['label'].value_counts()}")

---
## Step 2 - Balance & Split Data

In [ ]:
# Balance: equal samples per emotion
min_count = df['label'].value_counts().min()
df = df.groupby('label', group_keys=False).apply(
    lambda x: x.sample(n=min_count, random_state=42)
).reset_index(drop=True)

print(f"Balanced to {min_count} per label, total: {len(df)}")

# Explicit label mapping - this is the single source of truth
LABEL2ID = {"anger": 0, "fear": 1, "joy": 2, "love": 3, "sadness": 4, "surprise": 5}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

df['label_id'] = df['label'].map(LABEL2ID)

# Verify no unmapped labels
assert df['label_id'].isna().sum() == 0, "Some labels didn't match LABEL2ID! Check your CSV labels."

# 80/20 split on RAW text
X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['label_id'],
    test_size=0.2,
    stratify=df['label_id'],
    random_state=42
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

---
## Step 3 - Tokenize & Prepare Datasets

BERT has its own tokenizer. We feed it **raw text** — no stopword removal needed.

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_enc = tokenizer(list(X_train), truncation=True, padding=True, max_length=128)
test_enc  = tokenizer(list(X_test),  truncation=True, padding=True, max_length=128)

class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = EmotionDataset(train_enc, y_train)
test_dataset  = EmotionDataset(test_enc, y_test)

print("Datasets ready.")

---
## Step 4 - Train the Model

This takes ~15-25 minutes on a T4 GPU with the default dataset.

In [ ]:
from transformers import EarlyStoppingCallback

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=6,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {"accuracy": accuracy_score(p.label_ids, preds)}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,                # L2 regularization to prevent memorization
    warmup_ratio=0.1,                 # gradual LR ramp-up for stable training
    label_smoothing_factor=0.1,       # prevents overconfident predictions (key fix)
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", # select by lowest loss, not highest accuracy
    greater_is_better=False,           # lower loss = better
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],  # stop when val loss plateaus
)

trainer.train()

---
## Step 5 - Evaluate

In [ ]:
preds_output = trainer.predict(test_dataset)
y_pred = preds_output.predictions.argmax(-1)

print(classification_report(y_test, y_pred, target_names=list(LABEL2ID.keys())))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(LABEL2ID.keys()),
            yticklabels=list(LABEL2ID.keys()))
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

---
## Step 6 - Save Model to Google Drive

In [ ]:
model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

print(f"Model saved to {MODEL_DIR}")

---
## Step 7 - Load Saved Model & Predict

You can run everything from here downward to use an already-trained model without retraining.

In [ ]:
import torch
from transformers import BertForSequenceClassification, BertTokenizer

MODEL_DIR = "/content/drive/MyDrive/NeuroMera_Model"  # same path as above

model = BertForSequenceClassification.from_pretrained(MODEL_DIR)
tokenizer = BertTokenizer.from_pretrained(MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def predict_emotion(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()
    return model.config.id2label[pred]

print("Model loaded and ready.")

In [ ]:
# Quick test
tests = [
    "I love you so much",
    "I am not happy at all",
    "My heart is singing this morning",
    "I feel completely broken and alone",
    "That makes my blood boil",
    "I am shaking with fear",
    "I am extremely excited today",
]

for t in tests:
    print(f"{predict_emotion(t):>10}  |  {t}")

In [ ]:
# Interactive mode - type your own sentences
while True:
    user_input = input("\nEnter text (or 'exit'): ")
    if user_input.lower() == "exit":
        break
    print("Predicted Emotion:", predict_emotion(user_input))